In [27]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

library(mixOmics)

In [28]:
DATA_DIR <- "/home/mariya/Data/pallas"
TRAIN_DATA_DIR <- file.path(DATA_DIR, "nf-core-mlomix-train-bimodal/finalize")
TEST_DATA_DIR <- file.path(DATA_DIR, "nf-core-mlomix-test-bimodal/finalize")
METH_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.dnam.csv")
GEX_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.gex.csv")
LABELS_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "labels.csv")
METH_TEST_FILE <- file.path(TEST_DATA_DIR, "features.dnam.csv")
GEX_TEST_FILE <- file.path(TEST_DATA_DIR, "features.gex.csv")
LABELS_TEST_FILE <- file.path(TEST_DATA_DIR, "labels.csv")

In [29]:
# Make this into a function that can be invoked for both train and test sets

data <- function(gex_file, dnam_file, labels_file) {
    # --- Read in data ---
    X_gex  <- read.csv(gex_file,    row.names = 1, check.names = FALSE)
    X_dnam <- read.csv(dnam_file,   row.names = 1, check.names = FALSE)
    labels <- read.csv(labels_file, row.names = 1)

    # --- Extract Y as a factor ---
    # Replace "subtype" with your actual column name
    Y <- factor(labels[, "class"])

    # --- Align sample order across all three ---
    # (critical — mixOmics assumes rows are in the same order)
    common_samples <- Reduce(intersect, list(rownames(X_gex),
                                            rownames(X_dnam),
                                            rownames(labels)))

    X_gex  <- as.matrix(X_gex[common_samples, ])
    X_dnam <- as.matrix(X_dnam[common_samples, ])
    Y      <- Y[match(common_samples, rownames(labels))]

    # --- Sanity checks ---
    stopifnot(nrow(X_gex)  == length(Y))
    stopifnot(nrow(X_dnam) == length(Y))
    stopifnot(all(rownames(X_gex) == rownames(X_dnam)))

    cat("Samples:", length(Y), "\n")
    cat("GEX features:", ncol(X_gex), "\n")
    cat("DNAm features:", ncol(X_dnam), "\n")
    cat("Subtypes:", levels(Y), "\n")
    return(list(X_gex = X_gex, X_dnam = X_dnam, Y = Y))
}

train_data <- data(GEX_TRAIN_FILE, METH_TRAIN_FILE, LABELS_TRAIN_FILE)
test_data  <- data(GEX_TEST_FILE, METH_TEST_FILE, LABELS_TEST_FILE)

Samples: 827 
GEX features: 5000 
DNAm features: 5000 
Subtypes: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214 AML, ETV6-r AML, KAT6A-r AML, KMT2A-r AML, NPM1alt AML, NUP98-r AML, RUNX1::RUNX1T1 B-ALL, BCR::ABL1 B-ALL, BCR::ABL1-like B-ALL, DUX4-r B-ALL, ETV6::RUNX1 B-ALL, ETV6::RUNX1-like B-ALL, high hyperdiploid B-ALL, iAMP21 B-ALL, KMT2A-r B-ALL, MEF2D-r B-ALL, PAX5alt B-ALL, TCF3::PBX1 B-ALL, ZNF384-r Non-leukemic T-ALL 
Samples: 356 
GEX features: 5000 
DNAm features: 5000 
Subtypes: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214 AML, ETV6-r AML, KAT6A-r AML, KMT2A-r AML, NPM1alt AML, NUP98-r AML, RUNX1::RUNX1T1 B-ALL, BCR::ABL1 B-ALL, BCR::ABL1-like B-ALL, DUX4-r B-ALL, ETV6::RUNX1 B-ALL, ETV6::RUNX1-like B-ALL, high hyperdiploid B-ALL, iAMP21 B-ALL, KMT2A-r B-ALL, MEF2D-r B-ALL, PAX5alt B-ALL, TCF3::PBX1 B-ALL, ZNF384-r Non-leukemic T-ALL 


In [30]:
# Extract the following variables: X_gex_train, X_dnam_train, Y_train, X_gex_test, X_dnam_test, Y_test
X_gex_train <- train_data$X_gex
X_dnam_train <- train_data$X_dnam
Y_train     <- train_data$Y
X_gex_test  <- test_data$X_gex
X_dnam_test <- test_data$X_dnam
Y_test      <- test_data$Y

# Ensure test set has same columns in same order as train set
X_gex_test <- X_gex_test[, colnames(X_gex_train)]
X_dnam_test <- X_dnam_test[, colnames(X_dnam_train)]

head(X_gex_train)
head(X_dnam_train)
head(Y_train)
head(X_gex_test)
head(X_dnam_test)
head(Y_test)

,ENSG00000254772,ENSG00000196230,ENSG00000136167,ENSG00000120129,ENSG00000152518,ENSG00000108849,ENSG00000147127,ENSG00000130066,ENSG00000145741,ENSG00000116703,⋯,ENSG00000173207,ENSG00000139793,ENSG00000167797,ENSG00000110987,ENSG00000100162,ENSG00000142327,ENSG00000130770,ENSG00000168824,ENSG00000275066,ENSG00000150093
TARGET-20-PASWLN-D,-1.110315,-1.110315,-1.110315,-1.110315,-1.110315,7.968453,8.201993,-1.110315,-1.110315,7.629022,⋯,3.329529,6.377667,4.155871,5.503012,1.437385,3.416764,5.593546,1.5736289,5.615856,4.932075
TARGET-20-PAVNHN-D,-1.110315,-1.110315,-1.110315,-1.110315,-1.110315,9.651507,8.238651,-1.110315,-1.110315,8.743245,⋯,3.325243,4.998404,3.889626,2.388745,5.188508,6.753050,4.478985,-0.4569365,5.244575,4.588830
TARGET-20-PARURW-D,-1.110315,-1.110315,-1.110315,-1.110315,-1.110315,7.430994,9.591533,-1.110315,-1.110315,9.646101,⋯,6.390863,5.110798,6.692811,5.357277,4.683748,5.425402,8.064418,0.1519207,3.498843,4.158064
TARGET-20-PAPVKG-D,-1.110315,-1.110315,-1.110315,-1.110315,-1.110315,7.851423,9.648369,-1.110315,-1.110315,9.552824,⋯,7.075817,4.355816,6.616366,4.077949,5.673264,6.049784,7.412941,-0.1748666,4.167821,6.556703
TARGET-20-PASJDV-D,-1.110315,-1.110315,-1.110315,-1.110315,-1.110315,8.570422,9.675942,-1.110315,-1.110315,10.000632,⋯,4.323619,3.286130,6.892616,6.783589,3.792690,8.109676,7.042379,0.3849265,4.073624,6.144524
TARGET-20-PASAFH-D,-1.110315,-1.110315,-1.110315,-1.110315,-1.110315,8.277945,9.758931,-1.110315,-1.110315,10.104308,⋯,4.218665,5.559829,5.762059,6.228017,2.297075,6.620060,7.076211,0.8439528,4.406055,5.124707


,cg09698465,cg20439288,cg07110356,cg17353893,cg25670583,cg18559579,cg25588389,cg01030534,cg22301418,cg23478547,⋯,cg14293575,cg24563570,cg13931640,cg20722088,cg21531416,cg12103569,cg25970575,cg09187505,cg20165746,cg10725937
TARGET-20-PASWLN-D,0.9795,0.9637,0.9845,0.9739,0.9769,0.0258,0.9759,0.9759,0.9816,0.3976,⋯,0.9487,0.3689,0.9881,0.9521,0.9635,0.8681,0.2758,0.8247,0.0972,0.2172
TARGET-20-PAVNHN-D,0.4319,0.0347,0.0686,0.8751,0.3817,0.7416,0.0349,0.1404,0.5413,0.0383,⋯,0.6504,0.5236,0.8780,0.8355,0.8787,0.8327,0.7544,0.9197,0.1737,0.0928
TARGET-20-PARURW-D,0.9071,0.7292,0.1595,0.3500,0.2146,0.2064,0.1724,0.5086,0.2486,0.2907,⋯,0.8048,0.5735,0.3990,0.8588,0.4571,0.0973,0.4581,0.3174,0.8437,0.4610
TARGET-20-PAPVKG-D,0.5056,0.8414,0.0658,0.2899,0.1558,0.0576,0.2374,0.8892,0.1917,0.1229,⋯,0.9216,0.5348,0.3080,0.4343,0.4585,0.1123,0.6375,0.8231,0.8716,0.2129
TARGET-20-PASJDV-D,0.9710,0.8416,0.9605,0.9837,0.9580,0.9554,0.9494,0.8966,0.4994,0.9774,⋯,0.7699,0.7911,0.5653,0.9253,0.8863,0.0479,0.2803,0.9693,0.8632,0.4679
TARGET-20-PASAFH-D,0.8489,0.8338,0.0610,0.8793,0.0845,0.0668,0.9418,0.0619,0.0837,0.5277,⋯,0.4154,0.3757,0.3313,0.1368,0.3899,0.0702,0.0953,0.6072,0.7669,0.4770


[1] AML, KMT2A-r        AML, KMT2A-r        AML, RUNX1::RUNX1T1
[4] AML, RUNX1::RUNX1T1 AML, NUP98-r        AML, RUNX1::RUNX1T1
25 Levels: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 ... T-ALL

,ENSG00000254772,ENSG00000196230,ENSG00000136167,ENSG00000120129,ENSG00000152518,ENSG00000108849,ENSG00000147127,ENSG00000130066,ENSG00000145741,ENSG00000116703,⋯,ENSG00000173207,ENSG00000139793,ENSG00000167797,ENSG00000110987,ENSG00000100162,ENSG00000142327,ENSG00000130770,ENSG00000168824,ENSG00000275066,ENSG00000150093
TARGET-20-PAWCIL-D,-0.6804263,-0.6804263,-0.6804263,-0.6804263,-0.6804263,4.665046,7.976460,-0.6804263,-0.6804263,7.580697,⋯,5.711941,6.516569,4.252847,3.665822,4.211060,7.625540,6.035070,1.79101655,6.750193,7.606336
TARGET-20-PATAVF-D,-0.6804263,-0.6804263,-0.6804263,-0.6804263,-0.6804263,6.052405,6.804276,-0.6804263,-0.6804263,9.040013,⋯,7.130432,6.964781,6.100464,7.600587,5.756574,7.075772,5.790688,0.02011944,5.596456,10.430148
TARGET-20-PARNFZ-D,-0.6804263,-0.6804263,-0.6804263,-0.6804263,-0.6804263,4.212443,10.978106,-0.6804263,-0.6804263,7.310924,⋯,2.606330,6.092832,1.648707,7.102261,1.594647,6.517171,5.623885,2.70369024,9.683789,6.972729
TARGET-20-PAWWXU-D,-0.6804263,-0.6804263,-0.6804263,-0.6804263,-0.6804263,6.162425,7.648316,-0.6804263,-0.6804263,6.941058,⋯,3.717040,5.565006,5.368669,6.123119,3.349423,6.575066,6.504471,0.54608463,8.862040,8.299906
TARGET-20-PAVALK-D,-0.6804263,-0.6804263,-0.6804263,-0.6804263,-0.6804263,7.488189,7.691723,-0.6804263,-0.6804263,4.926455,⋯,2.975759,7.310202,3.960954,5.332682,5.179758,8.442750,4.564554,0.61702904,9.040386,6.549881
TARGET-20-PAWXCK-D,-0.6804263,-0.6804263,-0.6804263,-0.6804263,-0.6804263,7.176085,7.112607,-0.6804263,-0.6804263,7.800662,⋯,2.531455,8.317846,5.885741,3.414233,3.193770,8.483819,6.049334,5.04291644,5.930072,9.933570


,cg09698465,cg20439288,cg07110356,cg17353893,cg25670583,cg18559579,cg25588389,cg01030534,cg22301418,cg23478547,⋯,cg14293575,cg24563570,cg13931640,cg20722088,cg21531416,cg12103569,cg25970575,cg09187505,cg20165746,cg10725937
TARGET-20-PAWCIL-D,0.1951,0.9712,NA,0.3671,0.0813,0.1306,0.2840,0.0275,0.0720,NA,⋯,0.5601,NA,NA,NA,NA,0.4825,0.3464,0.7930,0.7946,0.1092
TARGET-20-PATAVF-D,0.9665,0.0144,0.0240,0.9631,0.0552,0.0360,0.9441,0.0360,0.6813,0.9663,⋯,0.9096,0.1640,0.4852,0.2521,0.7080,0.0401,0.9602,0.9119,0.2548,0.1565
TARGET-20-PARNFZ-D,0.8759,0.6837,0.1069,0.3485,0.2471,0.1656,0.1186,0.0397,0.3214,0.3301,⋯,0.8887,0.4487,0.4082,0.7098,0.6343,0.1065,0.1899,0.7905,0.8607,0.3387
TARGET-20-PAWWXU-D,0.7253,0.8489,0.0367,0.0462,0.0691,0.0339,0.0652,0.3549,0.0665,0.0454,⋯,0.8223,0.5591,0.2632,0.4683,0.5889,0.0679,0.6243,0.7367,0.6569,0.1680
TARGET-20-PAVALK-D,0.9601,0.0261,0.8369,0.9746,0.8747,0.9575,0.0888,0.0872,0.9293,0.0268,⋯,0.6117,0.0387,0.0835,0.7914,0.8988,0.4455,0.5309,0.0398,0.1007,0.0586
TARGET-20-PAWXCK-D,0.0004,0.0260,0.9171,0.9619,0.1737,0.3219,0.1195,0.9334,0.9638,0.0910,⋯,0.8474,0.5268,0.3184,0.8849,0.7671,0.8498,0.2004,0.5286,0.2017,0.5416


[1] AML, RUNX1::RUNX1T1 AML, CBFB::MYH11    AML, RUNX1::RUNX1T1
[4] AML, RUNX1::RUNX1T1 AML, KMT2A-r        AML, KMT2A-r       
25 Levels: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 ... T-ALL

In [31]:
# --- Evaluate: BER + per-class accuracy ---
ber <- function(true, pred) {
  classes <- levels(true)
  mean(sapply(classes, function(c) {
    idx <- true == c
    mean(pred[idx] != true[idx])
  }))
}

# --- Config ---
NCOMP <- 2
KEEPX <- 20


In [32]:

# --- GEX only ---
gex_model  <- splsda(X_gex_train, Y_train, ncomp = NCOMP, keepX = rep(KEEPX, NCOMP))
gex_pred   <- predict(gex_model, X_gex_test, dist = "max.dist")
gex_class  <- gex_pred$class$max.dist[, NCOMP]  # predictions at final component

round(ber(Y_test, gex_class),    3)

[1] 0.89

In [33]:
# --- DNAm only ---
dnam_model <- splsda(X_dnam_train, Y_train, ncomp = NCOMP, keepX = rep(KEEPX, NCOMP))
dnam_pred  <- predict(dnam_model, X_dnam_test, dist = "max.dist")
dnam_class <- dnam_pred$class$max.dist[, NCOMP]
round(ber(Y_test, dnam_class),   3)


[1] 0.881

In [34]:

# --- Bimodal DIABLO ---
design <- matrix(0.1, nrow = 2, ncol = 2,
                 dimnames = list(c("GEX", "DNAm"), c("GEX", "DNAm")))
diag(design) <- 0

diablo_model <- block.splsda(
  X = list(GEX = X_gex_train, DNAm = X_dnam_train),
  Y = Y_train, ncomp = NCOMP,
  keepX = list(GEX = rep(KEEPX, NCOMP), DNAm = rep(KEEPX, NCOMP)),
  design = design
)
diablo_pred  <- predict(diablo_model,
                        newdata = list(GEX = X_gex_test, DNAm = X_dnam_test),
                        dist = "max.dist")
diablo_class <- diablo_pred$WeightedVote$max.dist[, NCOMP]
round(ber(Y_test, diablo_class),    3)


Design matrix has changed to include Y; each block will be
            linked to Y.



[1] 0.883

In [ ]:

cat("=== Test Set BER ===\n")
cat("GEX only:  ", round(ber(Y_test, gex_class),    3), "\n")
cat("DNAm only: ", round(ber(Y_test, dnam_class),   3), "\n")
cat("Bimodal:   ", round(ber(Y_test, diablo_class), 3), "\n\n")

# --- Confusion matrices ---
cat("=== GEX Confusion Matrix ===\n");    print(table(Predicted = gex_class,    True = Y_test))
cat("=== DNAm Confusion Matrix ===\n");   print(table(Predicted = dnam_class,   True = Y_test))
cat("=== Bimodal Confusion Matrix ===\n");print(table(Predicted = diablo_class, True = Y_test))

=== Test Set BER ===
GEX only:   0.798 
DNAm only:  0.787 
Bimodal:    0.808 

=== GEX Confusion Matrix ===
                          True
Predicted                  AML, APL-PML::RARA AML, CBFA2T3::GLIS2
  AML, CBFB::MYH11                          0                   0
  AML, CEBPAalt                             0                   0
  AML, KMT2A-r                              0                   3
  AML, NPM1alt                              0                   1
  AML, RUNX1::RUNX1T1                       4                   0
  B-ALL, high hyperdiploid                  0                   0
                          True
Predicted                  AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214
  AML, CBFB::MYH11                       25             1                0
  AML, CEBPAalt                           0            18                0
  AML, KMT2A-r                            0             0                0
  AML, NPM1alt                            0             1           

=== DNAm Confusion Matrix ===
                          True
Predicted                  AML, APL-PML::RARA AML, CBFA2T3::GLIS2
  AML, CBFB::MYH11                          3                   1
  AML, CEBPAalt                             1                   0
  AML, KMT2A-r                              0                   2
  AML, NPM1alt                              0                   1
  AML, RUNX1::RUNX1T1                       0                   0
  B-ALL, high hyperdiploid                  0                   0
                          True
Predicted                  AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214
  AML, CBFB::MYH11                       22             0                0
  AML, CEBPAalt                           0            13                0
  AML, KMT2A-r                            0             6                0
  AML, NPM1alt                            0             1                8
  AML, RUNX1::RUNX1T1                     3             1              